# 特徴量エンジニアリング

欠損補完、カテゴリ変数エンコーディング、対数変換など

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("..")
from src.features.build_features import FeatureBuilder

train_raw = pd.read_csv("../data/raw/train.csv")
test_raw = pd.read_csv("../data/raw/test.csv")
train_raw.shape, test_raw.shape

((1460, 81), (1459, 80))

## 1. 外れ値除外(01_eda.ipynbで確認したGrLivArea外れ値)

In [2]:
outlier_mask = (train_raw["GrLivArea"] > 4000) & (train_raw["SalePrice"] < 300000)
print(f"除外する外れ値: {outlier_mask.sum()}件")
train_raw = train_raw[~outlier_mask].reset_index(drop=True)
train_raw.shape

除外する外れ値: 2件


(1458, 81)

## 2. 特徴量ビルド(train統計量でfit → train/testにtransform)

In [3]:
sale_price = train_raw["SalePrice"]
X_train_raw = train_raw.drop(columns=["SalePrice"])
X_test_raw = test_raw.copy()

fb = FeatureBuilder()
X_train = fb.fit_transform(X_train_raw)
X_test = fb.transform(X_test_raw)

X_train.shape, X_test.shape

((1458, 266), (1459, 266))

In [4]:
# 欠損が残っていないか確認
assert X_train.isnull().sum().sum() == 0, "X_trainに欠損が残っています"
assert X_test.isnull().sum().sum() == 0, "X_testに欠損が残っています"
print("欠損なしを確認")

欠損なしを確認


## 3. LightGBM向け: ネイティブカテゴリ用の特徴量ビルド

one-hotだと266列に展開されカテゴリの情報が疎になる。LightGBMは`categorical_feature`で
名義変数を直接カテゴリ分割できるため、one-hotせず整数コードのまま渡す版を別途作る。

In [5]:
fb_ord = FeatureBuilder(encoding="ordinal")
X_train_ord = fb_ord.fit_transform(X_train_raw)
X_test_ord = fb_ord.transform(X_test_raw)

assert X_train_ord.isnull().sum().sum() == 0, "X_train_ordに欠損が残っています"
assert X_test_ord.isnull().sum().sum() == 0, "X_test_ordに欠損が残っています"

X_train_ord.shape, X_test_ord.shape, fb_ord.nominal_cols_

((1458, 84),
 (1459, 84),
 ['MSZoning',
  'Street',
  'Alley',
  'LotShape',
  'LandContour',
  'Utilities',
  'LotConfig',
  'LandSlope',
  'Neighborhood',
  'Condition1',
  'Condition2',
  'BldgType',
  'HouseStyle',
  'RoofStyle',
  'RoofMatl',
  'Exterior1st',
  'Exterior2nd',
  'MasVnrType',
  'Foundation',
  'BsmtExposure',
  'BsmtFinType1',
  'BsmtFinType2',
  'Heating',
  'CentralAir',
  'Electrical',
  'Functional',
  'GarageType',
  'GarageFinish',
  'PavedDrive',
  'Fence',
  'MiscFeature',
  'SaleType',
  'SaleCondition'])

## 4. data/processed/ へ保存

`src/models/train.py` は `train_features.csv`(Ridge/Lasso向けone-hot)または
`train_features_lgbm.csv`(LightGBM向けordinal)から `SalePrice` を、
`src/models/predict.py` は対応する `test_features*.csv` から `Id` を読む前提。
`categorical_columns.json` はLightGBMの`categorical_feature`に渡す列名一覧。

In [6]:
import json

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_out = X_train.drop(columns=["Id"]).copy()
train_out["SalePrice"] = sale_price.values
train_out.to_csv(processed_dir / "train_features.csv", index=False)

test_out = X_test.copy()  # Idはpredict.pyがpopして提出csvに使う
test_out.to_csv(processed_dir / "test_features.csv", index=False)

train_out_ord = X_train_ord.drop(columns=["Id"]).copy()
train_out_ord["SalePrice"] = sale_price.values
train_out_ord.to_csv(processed_dir / "train_features_lgbm.csv", index=False)

test_out_ord = X_test_ord.copy()
test_out_ord.to_csv(processed_dir / "test_features_lgbm.csv", index=False)

with open(processed_dir / "categorical_columns.json", "w", encoding="utf-8") as f:
    json.dump(fb_ord.nominal_cols_, f, ensure_ascii=False)

print(f"train_features.csv: {train_out.shape}")
print(f"test_features.csv: {test_out.shape}")
print(f"train_features_lgbm.csv: {train_out_ord.shape}")
print(f"test_features_lgbm.csv: {test_out_ord.shape}")
print(f"categorical_columns.json: {len(fb_ord.nominal_cols_)}列")

train_features.csv: (1458, 266)
test_features.csv: (1459, 266)
train_features_lgbm.csv: (1458, 84)
test_features_lgbm.csv: (1459, 84)
categorical_columns.json: 33列
